# WDC Merged N-Quads — Full Corpus Analysis

Same assessment metrics as `wdc_analysis_by_class`, applied to `output/merged.nq` (all domains, all classes, deduplicated).

## 1. Setup

In [1]:
import re, json, requests, duckdb
from collections import defaultdict, Counter
from functools import lru_cache
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import statistics

SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
MERGED_FILE = Path("output/merged.nq")
DATA_DIR    = Path("wdc_class_data")
DB_PATH     = Path("output/store.duckdb")
DATA_DIR.mkdir(exist_ok=True)

assert MERGED_FILE.exists(), f"Run the merge cell in wdc_lookup_extractor.ipynb first"
print(f"Merged file : {MERGED_FILE}  ({MERGED_FILE.stat().st_size / 1e6:.1f} MB)")
print(f"DuckDB store: {DB_PATH}")

def normalize_schema_type(uri):
    return re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)


Merged file : output/merged.nq  (14359.5 MB)
DuckDB store: output/store.duckdb


## 2. Parse merged.nq → entity store

In [2]:
import os, tempfile, shutil
import pyarrow as pa
import pyarrow.parquet as pq

NQ_RE = re.compile(
    r'^'
    r'(<[^>]*>|_:\S+)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+'
    r'(<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@\w[\w-]*|(?:\^\^<[^>]+>))?)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+\.\s*$'
)

def strip_angle(s):
    return s[1:-1] if s.startswith('<') and s.endswith('>') else s


def build_duckdb_store(nq_path, db_path, chunk_rows=1_000_000):
    """
    Stream-parse nq → Parquet chunks (low RAM) → bulk-load into DuckDB.
    Peak memory: ~300 MB (one chunk of 1M rows in Python lists).
    """
    tmpdir = tempfile.mkdtemp(prefix='nq_parquet_')
    chunk_id = parse_errors = total = 0
    subj, pred, obj, graph = [], [], [], []

    def _flush():
        nonlocal chunk_id
        out = os.path.join(tmpdir, f'chunk_{chunk_id:04d}.parquet')
        pq.write_table(
            pa.table({'subject': subj, 'predicate': pred, 'object': obj, 'graph': graph}),
            out, compression='snappy'
        )
        chunk_id += 1
        subj.clear(); pred.clear(); obj.clear(); graph.clear()

    print(f'Parsing {nq_path.name} ...')
    with open(nq_path, encoding='utf-8', errors='replace') as f:
        for line in f:
            total += 1
            m = NQ_RE.match(line.rstrip('\n'))
            if not m:
                parse_errors += 1
                continue
            subj.append(strip_angle(m.group(1)))
            pred.append(strip_angle(m.group(2)))
            obj.append(m.group(3))
            graph.append(strip_angle(m.group(4)))
            if len(subj) >= chunk_rows:
                _flush()
            if total % 5_000_000 == 0:
                print(f'  {total:,} lines, {chunk_id} chunks written ...')
    if subj:
        _flush()
    print(f'  Done: {total:,} lines, {parse_errors:,} errors, {chunk_id} Parquet chunks')

    print('Loading chunks into DuckDB ...')
    parquet_glob = os.path.join(tmpdir, 'chunk_*.parquet')
    con = duckdb.connect(str(db_path))
    con.execute("SET memory_limit='3GB'")
    con.execute("SET preserve_insertion_order=false")
    con.execute("SET threads=4")
    con.execute(f"""
        CREATE OR REPLACE TABLE triples AS
        SELECT * FROM read_parquet('{parquet_glob}')
    """)
    shutil.rmtree(tmpdir)

    print('Building derived tables ...')
    con.execute(f"""
        CREATE OR REPLACE TABLE entity_types AS
        SELECT DISTINCT subject,
            CASE WHEN object LIKE '<%>' THEN substr(object, 2, length(object)-2)
                 ELSE object END AS type
        FROM triples WHERE predicate = '{RDF_TYPE}'
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE entity_pred AS
        SELECT DISTINCT subject, predicate, object
        FROM triples WHERE predicate != '{RDF_TYPE}'
    """)
    con.execute("""
        CREATE OR REPLACE TABLE entity_graphs AS
        SELECT DISTINCT subject, graph FROM triples
    """)
    con.execute("""
        CREATE OR REPLACE TABLE entity_refs AS
        WITH clean_obj AS (
            SELECT subject,
                CASE WHEN object LIKE '<%>' THEN substr(object, 2, length(object)-2)
                     ELSE object END AS linked_subject
            FROM entity_pred
            WHERE object LIKE '<%>' OR object LIKE '_:%'
        )
        SELECT DISTINCT subject, linked_subject
        FROM clean_obj
        WHERE linked_subject IN (SELECT DISTINCT subject FROM triples)
    """)
    print('Creating indexes ...')
    for sql in [
        'CREATE INDEX IF NOT EXISTS idx_et_subj ON entity_types(subject)',
        'CREATE INDEX IF NOT EXISTS idx_et_type ON entity_types(type)',
        'CREATE INDEX IF NOT EXISTS idx_ep_subj ON entity_pred(subject)',
        'CREATE INDEX IF NOT EXISTS idx_ep_pred ON entity_pred(predicate)',
        'CREATE INDEX IF NOT EXISTS idx_eg_graph ON entity_graphs(graph)',
        'CREATE INDEX IF NOT EXISTS idx_er_subj ON entity_refs(subject)',
    ]:
        con.execute(sql)
    # Completion marker — detected on next load to catch interrupted builds
    n_tri = con.execute('SELECT COUNT(*) FROM triples').fetchone()[0]
    con.execute("""
        CREATE OR REPLACE TABLE _build_meta AS
        SELECT CURRENT_TIMESTAMP AS built_at, ? AS n_triples
    """, [n_tri])
    n_ent = con.execute('SELECT COUNT(DISTINCT subject) FROM triples').fetchone()[0]
    print(f'  Triples  : {n_tri:,}')
    print(f'  Entities : {n_ent:,}')
    con.execute('CHECKPOINT')  # flush to main .duckdb file before returning
    return con, parse_errors, total


In [3]:
def _is_complete(con):
    tables = {r[0] for r in con.execute('SHOW TABLES').fetchall()}
    return '_build_meta' in tables

def _delete_store():
    for suffix in ['', '.wal', '.tmp']:
        f = DB_PATH.parent / (DB_PATH.name + suffix)
        try:
            if f.is_dir():
                shutil.rmtree(f)
                print(f'  Deleted {f.name}/ (dir)')
            elif f.exists():
                f.unlink()
                print(f'  Deleted {f.name}')
        except (PermissionError, OSError) as e:
            print(f'  Skipped {f.name} ({e})')

if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH))
    if _is_complete(con):
        n_ent = con.execute('SELECT COUNT(DISTINCT subject) FROM triples').fetchone()[0]
        n_tri = con.execute('SELECT COUNT(*) FROM triples').fetchone()[0]
        built_at = con.execute('SELECT built_at FROM _build_meta').fetchone()[0]
        print(f'Loaded DuckDB store  (built {built_at})')
        print(f'  Entities : {n_ent:,}')
        print(f'  Triples  : {n_tri:,}')
    else:
        print('Incomplete store detected — deleting and rebuilding ...')
        con.close()
        _delete_store()
        con, n_errors, n_total = build_duckdb_store(MERGED_FILE, DB_PATH)
        print(f'Done. DB size: {DB_PATH.stat().st_size / 1e9:.2f} GB')
else:
    print('No store found — building from scratch ...')
    con, n_errors, n_total = build_duckdb_store(MERGED_FILE, DB_PATH)
    print(f'Done. DB size: {DB_PATH.stat().st_size / 1e9:.2f} GB')


Loaded DuckDB store  (built 2026-04-27 14:59:39.941304+02:00)
  Entities : 12,211,353
  Triples  : 69,600,381


In [4]:
# import pathlib
# for f in pathlib.Path("output").glob("store.duckdb*"):
#     f.unlink()
#     print(f"Deleted {f}")


In [5]:
# (pickle helpers removed — caching is now handled by DuckDB store)


In [6]:
# (old cache-loading block removed — see cell above)


## 3. Load schema.org vocabulary

In [7]:
VOCAB_CACHE = DATA_DIR / "schemaorg_vocab.json"

def load_schemaorg_vocab():
    if VOCAB_CACHE.exists():
        print("Loading vocab from cache …")
        with open(VOCAB_CACHE) as f:
            raw = json.load(f)
    else:
        url = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
        print(f"Fetching {url} …")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.json()
        with open(VOCAB_CACHE, "w") as f:
            json.dump(raw, f)

    graph = raw.get("@graph", [])

    def local(uri):
        return uri.split("/")[-1].split("#")[-1].split(":")[-1]

    def ids_of(val):
        if isinstance(val, dict):  return [val.get("@id", "")]
        if isinstance(val, list):  return [v.get("@id", "") if isinstance(v, dict) else str(v) for v in val]
        return [str(val)]

    all_types    = set()
    type_props   = defaultdict(set)
    prop_range   = defaultdict(set)
    type_parents = defaultdict(set)

    for item in graph:
        item_type = str(item.get("@type", ""))
        if "rdfs:Class" in item_type or "schema:DataType" in item_type:
            name = local(item.get("@id", ""))
            all_types.add(name)
            for pid in ids_of(item.get("rdfs:subClassOf", [])):
                if pid: type_parents[name].add(local(pid))

    for item in graph:
        if "rdf:Property" in str(item.get("@type", "")):
            pname = local(item.get("@id", ""))
            for did in ids_of(item.get("schema:domainIncludes", [])):
                if did: type_props[local(did)].add(pname)
            for rid in ids_of(item.get("schema:rangeIncludes", [])):
                if rid: prop_range[pname].add(local(rid))

    memo = {}
    def get_ancestors(t):
        if t in memo: return memo[t]
        anc = set()
        for p in type_parents.get(t, set()):
            anc.add(p); anc |= get_ancestors(p)
        memo[t] = anc
        return anc

    for t in list(all_types):
        for anc in get_ancestors(t):
            type_props[t] |= type_props.get(anc, set())

    print(f"  Classes : {len(all_types):,}")
    return all_types, dict(type_props), dict(prop_range), dict(type_parents)

so_types, so_type_props, so_prop_range, so_type_parents = load_schemaorg_vocab()

# ── Upload vocab to DuckDB for SQL-assisted analyses ─────────────
type_prop_rows = [(t, p) for t, props in so_type_props.items() for p in props]
prop_range_rows = [(p, r) for p, ranges in so_prop_range.items() for r in ranges]
con.execute("CREATE OR REPLACE TABLE vocab_type_props (type TEXT, prop TEXT)")
con.executemany("INSERT INTO vocab_type_props VALUES (?, ?)", type_prop_rows)
con.execute("CREATE OR REPLACE TABLE vocab_prop_range (prop TEXT, range_type TEXT)")
con.executemany("INSERT INTO vocab_prop_range VALUES (?, ?)", prop_range_rows)
print(f"  Vocab uploaded: {len(type_prop_rows):,} type-prop pairs, {len(prop_range_rows):,} prop-range pairs")


Loading vocab from cache …
  Classes : 979
  Vocab uploaded: 63,436 type-prop pairs, 2,124 prop-range pairs


## 4. Namespace correctness

In [8]:
def namespace_report(con, label):
    def _counts(col, table):
        return con.execute(f"""
            SELECT
                SUM(CASE WHEN {col} LIKE 'http://schema.org/%'
                           OR {col} LIKE 'https://schema.org/%' THEN 1 ELSE 0 END) AS canonical,
                SUM(CASE WHEN {col} LIKE 'http://www.schema.org/%'
                           OR {col} LIKE 'https://www.schema.org/%' THEN 1 ELSE 0 END) AS non_canonical,
                SUM(CASE WHEN {col} NOT LIKE 'http://schema.org/%'
                          AND {col} NOT LIKE 'https://schema.org/%'
                          AND {col} NOT LIKE 'http://www.schema.org/%'
                          AND {col} NOT LIKE 'https://www.schema.org/%'
                          AND regexp_matches({col}, '(?i)schema\\.(org|org/)') THEN 1 ELSE 0 END) AS malformed,
                SUM(CASE WHEN NOT regexp_matches({col}, '(?i)schema\\.(org|org/)') THEN 1 ELSE 0 END) AS other
            FROM {table}
        """).fetchone()

    t = _counts('type', 'entity_types')
    p = _counts('predicate', '(SELECT DISTINCT subject, predicate FROM entity_pred)')

    canonical     = t[0] + p[0]
    non_canonical = t[1] + p[1]
    malformed     = t[2] + p[2]
    other         = t[3] + p[3]
    total_schema  = canonical + non_canonical + malformed
    pct_canonical     = canonical     / total_schema * 100 if total_schema else float('nan')
    pct_non_canonical = non_canonical / total_schema * 100 if total_schema else float('nan')

    www_ex = con.execute("""
        SELECT uri, COUNT(*) AS cnt FROM (
            SELECT type AS uri FROM entity_types
            WHERE type LIKE 'http://www.schema.org/%' OR type LIKE 'https://www.schema.org/%'
            UNION ALL
            SELECT predicate AS uri FROM (SELECT DISTINCT subject, predicate FROM entity_pred)
            WHERE predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
        ) GROUP BY uri ORDER BY cnt DESC LIMIT 5
    """).fetchall()

    mal_ex = con.execute("""
        SELECT uri, COUNT(*) AS cnt FROM (
            SELECT type AS uri FROM entity_types
            WHERE type NOT LIKE 'http://schema.org/%' AND type NOT LIKE 'https://schema.org/%'
              AND type NOT LIKE 'http://www.schema.org/%' AND type NOT LIKE 'https://www.schema.org/%'
              AND regexp_matches(type, '(?i)schema\\.(org|org/)')
            UNION ALL
            SELECT predicate AS uri FROM (SELECT DISTINCT subject, predicate FROM entity_pred)
            WHERE predicate NOT LIKE 'http://schema.org/%' AND predicate NOT LIKE 'https://schema.org/%'
              AND predicate NOT LIKE 'http://www.schema.org/%' AND predicate NOT LIKE 'https://www.schema.org/%'
              AND regexp_matches(predicate, '(?i)schema\\.(org|org/)')
        ) GROUP BY uri ORDER BY cnt DESC LIMIT 5
    """).fetchall()

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Canonical schema.org URIs        : {canonical:>10,}  ({pct_canonical:.2f}% of schema refs)")
    print(f"  Non-canonical (www.) URIs        : {non_canonical:>10,}  ({pct_non_canonical:.2f}% of schema refs)")
    print(f"  Truly malformed schema URIs      : {malformed:>10,}")
    print(f"  Other namespace URIs             : {other:>10,}  (rdf:, owl:, etc.)")
    if www_ex:
        print(f"\n  Non-canonical examples (top 5):")
        for uri, cnt in www_ex:
            print(f"    [{cnt:>5}]  {uri}")
    if mal_ex:
        print(f"\n  Truly malformed examples (top 5):")
        for uri, cnt in mal_ex:
            print(f"    [{cnt:>5}]  {uri}")
    return pct_canonical, pct_non_canonical

namespace_report(con, "All entities (merged.nq)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Canonical schema.org URIs        : 56,962,451  (99.90% of schema refs)
  Non-canonical (www.) URIs        :     57,157  (0.10% of schema refs)
  Truly malformed schema URIs      :          0
  Other namespace URIs             :    405,175  (rdf:, owl:, etc.)

  Non-canonical examples (top 5):
    [17372]  https://www.schema.org/SiteNavigationElement
    [13174]  http://www.schema.org/SiteNavigationElement
    [10352]  https://www.schema.org/name
    [10352]  https://www.schema.org/url
    [ 2954]  http://www.schema.org/url


(99.89975904429227, 0.10024095570772776)

schema.org doesn't use www

## 5. Type validity

In [9]:
def type_validity_report(con, label):
    rows = con.execute("""
        SELECT type, COUNT(*) AS cnt
        FROM entity_types
        WHERE type LIKE 'http://schema.org/%'     OR type LIKE 'https://schema.org/%'
           OR type LIKE 'http://www.schema.org/%' OR type LIKE 'https://www.schema.org/%'
        GROUP BY type ORDER BY cnt DESC
    """).fetchall()

    valid = Counter()
    invalid = Counter()
    for type_uri, cnt in rows:
        local_t = normalize_schema_type(type_uri)
        (valid if local_t in so_types else invalid)[local_t] += cnt

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float('nan')
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid types   : {sum(valid.values()):>10,}  ({rate:.2f}%)")
    print(f"  Invalid types : {sum(invalid.values()):>10,}  ({100-rate:.2f}%)")
    print(f"\n  Valid type distribution (top 20):")
    for t, c in valid.most_common(20):
        print(f"    {t:<45} {c:>9,}")
    if invalid:
        print(f"\n  Invalid types (top 10):")
        for t, c in invalid.most_common(10):
            print(f"    {t:<45} {c:>9,}")
    return rate, valid, invalid

type_rate, valid_types, invalid_types = type_validity_report(con, "All entities (merged.nq)")



────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Valid types   : 11,867,421  (99.76%)
  Invalid types :     28,102  (0.24%)

  Valid type distribution (top 20):
    ImageObject                                   1,746,687
    ListItem                                      1,509,100
    SportsTeam                                    1,045,713
    PostalAddress                                 1,030,247
    Organization                                  1,010,110
    Place                                           580,598
    Country                                         474,520
    AggregateRating                                 462,725
    SportsEvent                                     444,358
    Person                                          410,961
    SiteNavigationElement                           401,686
    BreadcrumbList                                  302,674
    WebPage     

## 6. Property validity

In [10]:
def prop_validity_report(con, label):
    @lru_cache(maxsize=None)
    def allowed_for_types(types_frozen):
        allowed = set()
        for t in types_frozen:
            allowed |= so_type_props.get(t, set())
        return allowed

    subj_types = defaultdict(set)
    for subj, t in con.execute("""
        SELECT subject, type FROM entity_types
        WHERE type LIKE 'http://schema.org/%'     OR type LIKE 'https://schema.org/%'
           OR type LIKE 'http://www.schema.org/%' OR type LIKE 'https://www.schema.org/%'
    """).fetchall():
        subj_types[subj].add(normalize_schema_type(t))

    subj_allowed = {s: allowed_for_types(frozenset(ts)) for s, ts in subj_types.items()}

    valid_props = Counter()
    invalid_props = Counter()
    for subj, pred in con.execute("""
        SELECT DISTINCT subject, predicate FROM entity_pred
        WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
           OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
    """).fetchall():
        p = normalize_schema_type(pred)
        (valid_props if p in subj_allowed.get(subj, set()) else invalid_props)[p] += 1

    total = sum(valid_props.values()) + sum(invalid_props.values())
    rate  = sum(valid_props.values()) / total * 100 if total else float('nan')
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid property uses   : {sum(valid_props.values()):>10,}  ({rate:.2f}%)")
    print(f"  Invalid property uses : {sum(invalid_props.values()):>10,}  ({100-rate:.2f}%)")
    print(f"\n  Most used valid properties (top 20):")
    for p, c in valid_props.most_common(20):
        print(f"    {p:<45} {c:>9,}")
    if invalid_props:
        print(f"\n  Invalid properties (top 10):")
        for p, c in invalid_props.most_common(10):
            print(f"    {p:<45} {c:>9,}")
    return rate, valid_props, invalid_props

prop_rate, valid_props, invalid_props = prop_validity_report(con, "All entities (merged.nq)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Valid property uses   : 44,753,999  (99.18%)
  Invalid property uses :    370,086  (0.82%)

  Most used valid properties (top 20):
    name                                          6,578,093
    url                                           4,920,617
    height                                        1,644,399
    width                                         1,615,246
    position                                      1,512,667
    image                                         1,421,062
    address                                       1,283,189
    description                                   1,179,263
    contentUrl                                    1,104,876
    item                                          1,017,368
    addressLocality                               1,005,532
    addressRegion                                   945,4

fix the above (url, name, description, etc. appear as invalid because the entity itself has no recognized schema.org type "maybe malformed schema.org uri" / not because those types lack url) / check them further

In [11]:
## 6b. Why are invalid properties invalid?

def invalid_prop_reason_report(con, subj_allowed, invalid_props, label):
    all_so_props = set()
    for props in so_type_props.values():
        all_so_props |= props

    reason_counts = Counter()
    per_prop_reasons = defaultdict(lambda: Counter())

    for subj, pred in con.execute("""
        SELECT DISTINCT subject, predicate FROM entity_pred
        WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
           OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
    """).fetchall():
        p = normalize_schema_type(pred)
        allowed = subj_allowed.get(subj, set())
        if p in allowed:
            continue

        if subj not in subj_allowed:
            reason = 'untyped_entity'   # no schema.org type triple at all
        elif not allowed:
            reason = 'unknown_type'      # has schema.org type but not in vocabulary (incl. wrong case)
        elif p not in all_so_props:
            reason = 'unknown_property'
        else:
            reason = 'domain_mismatch'

        reason_counts[reason] += 1
        per_prop_reasons[p][reason] += 1

    total_invalid = sum(reason_counts.values())
    print(f"\n{'\u2550'*65}")
    print(f"  {label} \u2014 Why are properties flagged as invalid?")
    print(f"{'\u2550'*65}")
    for reason, cnt in reason_counts.most_common():
        pct = cnt / total_invalid * 100 if total_invalid else 0
        print(f"  {reason:<30} {cnt:>10,}  ({pct:.1f}%)")

    print(f"\n  Top invalid props broken down by reason:")
    print(f"  {'property':<30} {'total':>8}  untyped  unk_type  unknown  domain_mismatch")
    print(f"  {'-'*80}")
    for p, total in invalid_props.most_common(20):
        r = per_prop_reasons[p]
        print(f"  {p:<30} {total:>8,}  {r['untyped_entity']:>7,}  {r['unknown_type']:>8,}  {r['unknown_property']:>7,}  {r['domain_mismatch']:>15,}")

    print(f"\n  Domain-mismatch detail (which entity types use props outside their domain):")
    mismatch_type_prop = Counter()
    for subj, pred in con.execute("""
        SELECT DISTINCT subject, predicate FROM entity_pred
        WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
           OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
    """).fetchall():
        p = normalize_schema_type(pred)
        allowed = subj_allowed.get(subj, set())
        if p in allowed or subj not in subj_allowed or not allowed:
            continue
        if p not in all_so_props:
            continue
        for t in con.execute("""
            SELECT type FROM entity_types WHERE subject = ?
        """, [subj]).fetchall():
            local_t = normalize_schema_type(t[0])
            mismatch_type_prop[(local_t, p)] += 1

    print(f"  {'entity_type':<30} {'property':<30} {'count':>8}")
    print(f"  {'-'*72}")
    for (t, p), cnt in mismatch_type_prop.most_common(30):
        print(f"  {t:<30} {p:<30} {cnt:>8,}")

    return reason_counts, per_prop_reasons


from functools import lru_cache

@lru_cache(maxsize=None)
def _allowed_for_types(types_frozen):
    allowed = set()
    for t in types_frozen:
        allowed |= so_type_props.get(t, set())
    return allowed

_subj_types = defaultdict(set)
for _subj, _t in con.execute("""
    SELECT subject, type FROM entity_types
    WHERE type LIKE 'http://schema.org/%'     OR type LIKE 'https://schema.org/%'
       OR type LIKE 'http://www.schema.org/%' OR type LIKE 'https://www.schema.org/%'
""").fetchall():
    _subj_types[_subj].add(normalize_schema_type(_t))

subj_allowed = {s: _allowed_for_types(frozenset(ts)) for s, ts in _subj_types.items()}

reason_counts, per_prop_reasons = invalid_prop_reason_report(
    con, subj_allowed, invalid_props, "All entities (merged.nq)"
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


═════════════════════════════════════════════════════════════════
  All entities (merged.nq) — Why are properties flagged as invalid?
═════════════════════════════════════════════════════════════════
  untyped_entity                    115,149  (31.1%)
  unknown_type                      101,467  (27.4%)
  unknown_property                   93,779  (25.3%)
  domain_mismatch                    59,691  (16.1%)

  Top invalid props broken down by reason:
  property                          total  untyped  unk_type  unknown  domain_mismatch
  --------------------------------------------------------------------------------
  url                              69,754   52,232    17,522        0                0
  name                             69,619   59,167    10,452        0                0
  price                            48,464       98         0        0           48,366
  query-input                      24,424        0         0   24,424                0
  width                  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  entity_type                    property                          count
  ------------------------------------------------------------------------
  MedicalProcedure               price                            48,366
  Person                         width                             2,615
  Article                        primaryImageOfPage                1,948
  NewsArticle                    primaryImageOfPage                1,948
  Organization                   jobTitle                          1,813
  Organization                   publisher                           769
  PerformanceRole                actor                               227
  TouristAttraction              author                              210
  TouristAttraction              reviewBody                          170
  Event                          address                             156
  Event                          geo                                 156
  Recipe                         articleSection  

## 7. Range violations

In [12]:
DATE_PROPS = {"datePublished", "dateModified", "dateCreated", "expires", "startDate", "endDate", "uploadDate"}
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean", "Date", "DateTime", "Time", "URL", "Duration"}

ISO_DATE_RE = re.compile(
    r'^\d{4}(?:-\d{2}(?:-\d{2}(?:T\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?(?:Z|[+-]\d{2}:\d{2})?)?)?)?$'
)
LANG_RE = re.compile(r"^[a-zA-Z]{2,3}(-[a-zA-Z0-9]{2,8})*$")

def is_literal(obj): return obj.startswith('"')
def is_uri(obj):     return obj.startswith('<')
def is_blank(obj):   return obj.startswith('_:')

def strip_literal(obj):
    m = re.match(r'^"((?:[^"\\]|\\.)*)', obj)
    return m.group(1) if m else obj

def validate_literal(val, ranges, prop):
    val = val.strip()
    if "Text" in ranges: return True
    if "URL" in ranges: return val.startswith("http")
    if "Boolean" in ranges: return val in {"true", "false", "0", "1"}
    if "Integer" in ranges: return re.fullmatch(r"-?\d+", val) is not None
    if "Float" in ranges or "Number" in ranges: return re.fullmatch(r"-?\d+(\.\d+)?", val) is not None
    if "Date" in ranges or "DateTime" in ranges:
        return ISO_DATE_RE.match(val.replace(" ", "T")) is not None
    if prop == "inLanguage": return LANG_RE.match(val) is not None
    return False

def range_violation_report(con, label):
    violations = Counter()
    ok = Counter()
    viol_examples = defaultdict(list)

    cur = con.execute("""
        SELECT predicate, object FROM entity_pred
        WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
           OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
    """)
    while True:
        batch = cur.fetchmany(20_000)
        if not batch:
            break
        for pred, obj in batch:
            prop_local    = normalize_schema_type(pred)
            ranges        = so_prop_range.get(prop_local, set())
            literal_ranges = ranges & LITERAL_DATATYPES
            entity_ranges  = ranges - LITERAL_DATATYPES
            valid = False
            if is_literal(obj):
                if literal_ranges:
                    valid = validate_literal(strip_literal(obj), literal_ranges, prop_local)
            elif is_uri(obj) or is_blank(obj):
                valid = bool(entity_ranges) or ("URL" in literal_ranges)
            if valid:
                ok[prop_local] += 1
            else:
                violations[prop_local] += 1
                if len(viol_examples[prop_local]) < 3:
                    viol_examples[prop_local].append(
                        f"lit={strip_literal(obj)[:60]!r}" if is_literal(obj) else f"uri={obj[:60]!r}"
                    )

    total = sum(violations.values()) + sum(ok.values())
    viol_rate = sum(violations.values()) / total * 100 if total else float('nan')
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Checked value assertions : {total:>10,}")
    print(f"  Range violations         : {sum(violations.values()):>10,}  ({viol_rate:.2f}%)")
    print(f"\n  Top violation properties (top 15):")
    for p, c in violations.most_common(15):
        print(f"    {p:<35} {c:>8,}  e.g. {viol_examples.get(p, [])}")
    return viol_rate, violations

range_rate, violations = range_violation_report(con, "All entities (merged.nq)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Checked value assertions : 55,056,887
  Range violations         :  6,054,576  (11.00%)

  Top violation properties (top 15):
    height                              1,640,370  e.g. ["lit='60'", "lit='675'", "lit='60'"]
    width                               1,613,832  e.g. ["lit='600'", "lit='600'", "lit='600'"]
    item                                 611,303  e.g. ["lit='https://immobilier.lefigaro.fr/'", "lit='https://immobilier.lefigaro.fr/prix-immobilier/roanne/ville-'", "lit='https://immobilier.lefigaro.fr/prix-immobilier'"]
    endDate                              395,515  e.g. ["lit='2019-06-10T16:00:00-0700'", "lit=''", "lit=''"]
    startDate                            382,002  e.g. ["lit='2022-01-23T23:05:00-0500'", "lit='        \\\\n    Sep 14, 2024'", "lit='        \\\\n    Oct 12, 2024'"]
    eventStatus                

## 8. Prevalence — entities per domain

In [13]:
def _get_pld(url):
    try:
        host  = urlparse(url).netloc or url
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return url

def prevalence_report(con, label, top_n=30):
    rows = con.execute("""
        SELECT DISTINCT subject, graph FROM entity_graphs
    """).fetchall()

    domain_counts = Counter()
    seen = defaultdict(set)
    for subj, graph in rows:
        d = _get_pld(graph)
        if d not in seen[subj]:
            seen[subj].add(d)
            domain_counts[d] += 1

    total = sum(domain_counts.values())
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Total entity-domain pairs : {total:,}")
    print(f"  Unique domains            : {len(domain_counts):,}")
    print(f"\n  Top {top_n} domains by entity count:")
    df = pd.DataFrame(domain_counts.most_common(top_n), columns=["domain", "entities"])
    df["% of total"] = (df["entities"] / total * 100).round(2)
    print(df.to_string(index=False))
    return domain_counts

domain_counts = prevalence_report(con, "All entities (merged.nq)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Total entity-domain pairs : 12,211,366
  Unique domains            : 112

  Top 30 domains by entity count:
             domain  entities  % of total
             ria.ru   3638349       29.79
         elpais.com   1301395       10.66
      cbssports.com   1221854       10.01
          sports.ru    723710        5.93
freepressjournal.in    675231        5.53
     indiatimes.com    503533        4.12
             rbc.ua    408279        3.34
        lefigaro.fr    342138        2.80
             as.com    340609        2.79
            cnn.com    281225        2.30
      wikipedia.org    277768        2.27
             aif.ru    266521        2.18
        cbsnews.com    251695        2.06
      elespanol.com    248921        2.04
             dw.com    208339        1.71
             com.br    175281        1.44
           idnes.cz    138

In [14]:
def _parse_host(uri):
    try:
        host = urlparse(uri).netloc.lower()
        parts = host.split(".")
        if len(parts) >= 3 and parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            reg = ".".join(parts[-3:])
        else:
            reg = ".".join(parts[-2:]) if len(parts) >= 2 else host
        return host, reg
    except Exception:
        return None, None

page_rows = con.execute("""
    SELECT eg.graph, COUNT(DISTINCT et.type) AS n_types
    FROM entity_graphs eg
    LEFT JOIN entity_types et
        ON eg.subject = et.subject
        AND (   et.type LIKE 'http://schema.org/%'
             OR et.type LIKE 'https://schema.org/%'
             OR et.type LIKE 'http://www.schema.org/%'
             OR et.type LIKE 'https://www.schema.org/%')
    GROUP BY eg.graph
""").fetchall()

page_type_counts = []
for graph, n_types in page_rows:
    _, reg = _parse_host(graph)
    page_type_counts.append({"uri": graph, "domain": reg, "n_types": n_types})

df_pt = pd.DataFrame(page_type_counts)

print(f"{'='*65}")
print("Avg distinct schema.org types per page — by domain")
print(f"{'='*65}")
print(f"  {'domain':<35} {'pages':>7}  {'avg_types':>10}  {'median':>7}  {'max':>5}")
print(f"  {'─'*63}")

summary = (
    df_pt.groupby("domain")
    .agg(
        pages     = ("uri",     "count"),
        avg_types = ("n_types", "mean"),
        median    = ("n_types", "median"),
        max_types = ("n_types", "max"),
    )
    .sort_values("pages", ascending=False)
)
for domain, row in summary.iterrows():
    print(f"  {domain:<35} {row['pages']:>7,}  {row['avg_types']:>10.2f}  "
          f"{row['median']:>7.1f}  {row['max_types']:>5.0f}")

print(f"\n  {'OVERALL':<35} {len(df_pt):>7,}  {df_pt['n_types'].mean():>10.2f}  "
      f"{df_pt['n_types'].median():>7.1f}  {df_pt['n_types'].max():>5}")

print(f"\n{'='*60}")
print("Distribution of type count per page (all domains)")
print(f"{'='*60}")
dist = Counter(df_pt["n_types"].astype(int))
bar_max = max(dist.values())
for k in sorted(dist):
    bar = "█" * (dist[k] * 40 // bar_max)
    print(f"  {k:>3} types: {bar:<40} {dist[k]:>8,}  ({dist[k]/len(df_pt)*100:>5.1f}%)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Avg distinct schema.org types per page — by domain
  domain                                pages   avg_types   median    max
  ───────────────────────────────────────────────────────────────
  ria.ru                              101,258.0        9.84     10.0     18
  wikipedia.org                       66,735.0        3.00      3.0      3
  dw.com                              41,245.0        4.51      5.0      5
  sports.ru                           38,174.0        2.85      3.0      7
  elpais.com                          36,821.0        7.77      7.0     29
  indiatimes.com                      33,332.0        8.40      9.0     16
  aif.ru                              30,720.0        6.40      7.0      7
  rbc.ua                              30,542.0       10.00     10.0     11
  lefigaro.fr                         24,615.0        8.61      9.0     10
  cnn.com                             18,825.0        7.81      8.0     15
  cbssports.com                       17,983.0        4.09

## 9. Property coverage — per top schema.org type

In [15]:
def coverage_report(con, target_class):
    defined_props = so_type_props.get(target_class, set())
    if not defined_props:
        print(f"  No properties defined for {target_class}")
        return

    so_prefix = (
        "http://schema.org/"     + target_class,
        "https://schema.org/"    + target_class,
        "http://www.schema.org/" + target_class,
        "https://www.schema.org/"+ target_class,
    )
    n = con.execute("""
        SELECT COUNT(DISTINCT subject) FROM entity_types
        WHERE type = ? OR type = ? OR type = ? OR type = ?
    """, so_prefix).fetchone()[0]
    if not n:
        print(f"  No entities of type {target_class}")
        return

    prop_freq = Counter()
    rows = con.execute("""
        SELECT ep.predicate, COUNT(DISTINCT ep.subject) AS cnt
        FROM entity_pred ep
        JOIN entity_types et ON ep.subject = et.subject
        WHERE (et.type = ? OR et.type = ? OR et.type = ? OR et.type = ?)
          AND (ep.predicate LIKE 'http://schema.org/%'     OR ep.predicate LIKE 'https://schema.org/%'
            OR ep.predicate LIKE 'http://www.schema.org/%' OR ep.predicate LIKE 'https://www.schema.org/%')
        GROUP BY ep.predicate
    """, so_prefix).fetchall()
    for pred, cnt in rows:
        p = normalize_schema_type(pred)
        if p in defined_props:
            prop_freq[p] = cnt

    used     = set(prop_freq.keys())
    coverage = len(used) / len(defined_props) * 100

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Property Coverage  ({n:,} entities)")
    print(f"{'─'*60}")
    print(f"  Properties defined : {len(defined_props):>5}")
    print(f"  Properties used    : {len(used):>5}  ({coverage:.1f}%)")
    print(f"  Properties unused  : {len(defined_props)-len(used):>5}  ({100-coverage:.1f}%)")

    data_rows = [(p, prop_freq.get(p, 0), prop_freq.get(p, 0)/n*100) for p in defined_props]
    data_rows.sort(key=lambda x: -x[1])
    df = pd.DataFrame(data_rows, columns=["property", "entities_using", "density_%"])
    df["density_%"] = df["density_%"].round(2)
    print(f"\n  Properties used (sorted by frequency):")
    print(df[df["entities_using"] > 0].to_string(index=False))
    unused = sorted(defined_props - used)
    print(f"\n  Never-used: {unused}")

TOP_TYPES = [t for t, _ in valid_types.most_common(10)]
print(f"Running coverage for top types: {TOP_TYPES}\n")
for cls in TOP_TYPES:
    coverage_report(con, cls)


Running coverage for top types: ['ImageObject', 'ListItem', 'SportsTeam', 'PostalAddress', 'Organization', 'Place', 'Country', 'AggregateRating', 'SportsEvent', 'Person']



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  ImageObject — Property Coverage  (1,746,687 entities)
────────────────────────────────────────────────────────────
  Properties defined :   151
  Properties used    :    26  (17.2%)
  Properties unused  :   125  (82.8%)

  Properties used (sorted by frequency):
            property  entities_using  density_%
               width          959172      54.91
                 url          706569      40.45
              height          653878      37.44
representativeOfPage          190287      10.89
          contentUrl          134495       7.70
  sourceOrganization           47282       2.71
         dateCreated           45003       2.58
        dateModified           43926       2.51
                name           35168       2.01
             caption           28199       1.61
          creditText           27794       1.59
       datePublished           20329       1.16
         description           19750       1.13
  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  ListItem — Property Coverage  (1,509,100 entities)
────────────────────────────────────────────────────────────
  Properties defined :    17
  Properties used    :    10  (58.8%)
  Properties unused  :     7  (41.2%)

  Properties used (sorted by frequency):
      property  entities_using  density_%
          item          876133      58.06
           url          770985      51.09
          name          140156       9.29
      position          140155       9.29
         image           10235       0.68
additionalType             314       0.02
  previousItem             145       0.01
   description              19       0.00
      nextItem               2       0.00
        sameAs               1       0.00

  Never-used: ['alternateName', 'disambiguatingDescription', 'identifier', 'mainEntityOfPage', 'owner', 'potentialAction', 'subjectOf']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  SportsTeam — Property Coverage  (1,045,713 entities)
────────────────────────────────────────────────────────────
  Properties defined :    93
  Properties used    :    19  (20.4%)
  Properties unused  :    74  (79.6%)

  Properties used (sorted by frequency):
        property  entities_using  density_%
           image          102597       9.81
             url           72938       6.97
            name           31038       2.97
        memberOf            5403       0.52
           coach             963       0.09
foundingLocation             727       0.07
    foundingDate             715       0.07
         athlete             427       0.04
           sport             252       0.02
            logo             160       0.02
          gender             116       0.01
          member             112       0.01
         address               1       0.00
          sameAs               1       0.00
       legalNam

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  Organization — Property Coverage  (1,010,110 entities)
────────────────────────────────────────────────────────────
  Properties defined :    89
  Properties used    :    33  (37.1%)
  Properties unused  :    56  (62.9%)

  Properties used (sorted by frequency):
                property  entities_using  density_%
                    logo          716434      70.93
                     url          580306      57.45
                  sameAs          508015      50.29
                 address          484881      48.00
               legalName          456824      45.23
                   email          456701      45.21
               telephone          456334      45.18
         aggregateRating          456154      45.16
                   award          456071      45.15
                   image          318938      31.57
                    name           51381       5.09
             description           12555       1.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  SportsEvent — Property Coverage  (444,358 entities)
────────────────────────────────────────────────────────────
  Properties defined :    61
  Properties used    :    17  (27.9%)
  Properties unused  :    44  (72.1%)

  Properties used (sorted by frequency):
           property  entities_using  density_%
               name          432904      97.42
           location          432731      97.38
            endDate          403573      90.82
        description          403569      90.82
              sport          238933      53.77
           homeTeam          166531      37.48
eventAttendanceMode          155902      35.08
          startDate           11194       2.52
           awayTeam            8949       2.01
                url            6630       1.49
         competitor            5213       1.17
          performer            4725       1.06
          organizer            2836       0.64
              imag

## 10. Average properties per entity (AvP)

In [16]:
def avp_report(con, label, filter_type=None):
    if filter_type:
        so_prefix = (
            "http://schema.org/"     + filter_type,
            "https://schema.org/"    + filter_type,
            "http://www.schema.org/" + filter_type,
            "https://www.schema.org/"+ filter_type,
        )
        prop_counts = [r[0] for r in con.execute("""
            SELECT COUNT(DISTINCT ep.predicate) AS n
            FROM entity_types et
            JOIN entity_pred ep ON et.subject = ep.subject
            WHERE (et.type = ? OR et.type = ? OR et.type = ? OR et.type = ?)
              AND (ep.predicate LIKE 'http://schema.org/%'     OR ep.predicate LIKE 'https://schema.org/%'
                OR ep.predicate LIKE 'http://www.schema.org/%' OR ep.predicate LIKE 'https://www.schema.org/%')
            GROUP BY et.subject
        """, so_prefix).fetchall()]
    else:
        prop_counts = [r[0] for r in con.execute("""
            SELECT COUNT(DISTINCT predicate) AS n
            FROM entity_pred
            WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
               OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
            GROUP BY subject
        """).fetchall()]
        n_total = con.execute("SELECT COUNT(DISTINCT subject) FROM entity_graphs").fetchone()[0]
        prop_counts += [0] * (n_total - len(prop_counts))

    if not prop_counts:
        print(f"  No entities for {label}"); return

    avp = statistics.mean(prop_counts)
    med = statistics.median(prop_counts)
    std = statistics.stdev(prop_counts) if len(prop_counts) > 1 else 0
    mn, mx = min(prop_counts), max(prop_counts)

    print(f"\n{'─'*60}")
    print(f"  {label} — Average Properties per Entity (AvP)")
    print(f"{'─'*60}")
    print(f"  Entities        : {len(prop_counts):>10,}")
    print(f"  AvP (mean)      : {avp:>10.2f}")
    print(f"  Median          : {med:>10.1f}")
    print(f"  Std dev         : {std:>10.2f}")
    print(f"  Min / Max       : {mn:>4}  /  {mx:>4}")

    buckets = Counter(min(c, 20) for c in prop_counts)
    print(f"\n  Distribution:")
    bar_max = max(buckets.values())
    for k in range(0, min(mx + 2, 22)):
        cnt = buckets.get(k, 0)
        bar = '█' * (cnt * 40 // bar_max)
        label_k = f"{k:>2}" if k < 20 else "20+"
        print(f"    {label_k}: {bar:<40} {cnt:>9,}  ({cnt/len(prop_counts)*100:>5.1f}%)")

avp_report(con, "All entities (merged.nq)")

print(f"\n\n{'═'*60}")
print("  AvP breakdown by schema.org type")
print(f"{'═'*60}")
print(f"  {'type':<35} {'entities':>9}  {'avail':>6}  {'AvP':>6}  {'median':>7}")
print(f"  {'─'*65}")
for cls, _ in valid_types.most_common(20):
    so_prefix = (
        "http://schema.org/"     + cls,
        "https://schema.org/"    + cls,
        "http://www.schema.org/" + cls,
        "https://www.schema.org/"+ cls,
    )
    counts = [r[0] for r in con.execute("""
        SELECT COUNT(DISTINCT ep.predicate) AS n
        FROM entity_types et
        JOIN entity_pred ep ON et.subject = ep.subject
        WHERE (et.type = ? OR et.type = ? OR et.type = ? OR et.type = ?)
          AND (ep.predicate LIKE 'http://schema.org/%'     OR ep.predicate LIKE 'https://schema.org/%'
            OR ep.predicate LIKE 'http://www.schema.org/%' OR ep.predicate LIKE 'https://www.schema.org/%')
        GROUP BY et.subject
    """, so_prefix).fetchall()]
    if not counts: continue
    n_defined = len(so_type_props.get(cls, set()))
    print(f"  {cls:<35} {len(counts):>9,}  {n_defined:>6}  {statistics.mean(counts):>6.2f}  {statistics.median(counts):>7.1f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  All entities (merged.nq) — Average Properties per Entity (AvP)
────────────────────────────────────────────────────────────
  Entities        : 12,211,353
  AvP (mean)      :       3.70
  Median          :        3.0
  Std dev         :       3.60
  Min / Max       :    0  /    31

  Distribution:
     0: ███████                                    460,708  (  3.8%)
     1: ████████████████████████████████████████ 2,605,661  ( 21.3%)
     2: ███████████████████████████████████████  2,601,181  ( 21.3%)
     3: ███████████████████████████              1,781,355  ( 14.6%)
     4: ████████████████████████████             1,862,873  ( 15.3%)
     5: ███████████████                            983,368  (  8.1%)
     6: ████                                       283,701  (  2.3%)
     7: ██                                         150,826  (  1.2%)
     8: ████                                       297,050  (  2.4%)
     9: ████    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ImageObject                         1,743,504     151    3.73      4.0
  ListItem                            1,509,096      17    2.75      3.0
  SportsTeam                          1,042,485      93    1.19      1.0
  PostalAddress                       1,008,318      30    4.51      5.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Organization                        1,009,766      89    5.92      4.0
  Place                                 580,546      60    2.02      2.0
  Country                               474,520      60    1.00      1.0
  AggregateRating                       462,696      22    3.99      4.0
  SportsEvent                           444,356      61    8.13      8.0
  Person                                410,829      81    2.79      2.0
  SiteNavigationElement                 384,436     131    1.90      2.0
  BreadcrumbList                        302,673      17    1.00      1.0
  WebPage                               140,079     139    4.59      3.0
  LocalBusiness                         214,643     127    3.71      3.0
  NewsArticle                           199,809     141   12.61     14.0
  NewsMediaOrganization                 184,805      93    5.04      3.0
  Article                               172,671     136   19.57     25.0
  WebSite                               126,518    

## 11. Characteristic property sets


In [17]:
def characteristic_sets_report(con, target_class, label=None, top_n=20):
    if label is None:
        label = target_class
    so_prefix = (
        "http://schema.org/"     + target_class,
        "https://schema.org/"    + target_class,
        "http://www.schema.org/" + target_class,
        "https://www.schema.org/"+ target_class,
    )

    rows = con.execute("""
        SELECT ep.subject, ep.predicate
        FROM entity_types et
        JOIN entity_pred ep ON et.subject = ep.subject
        WHERE (et.type = ? OR et.type = ? OR et.type = ? OR et.type = ?)
          AND (ep.predicate LIKE 'http://schema.org/%'     OR ep.predicate LIKE 'https://schema.org/%'
            OR ep.predicate LIKE 'http://www.schema.org/%' OR ep.predicate LIKE 'https://www.schema.org/%')
    """, so_prefix).fetchall()

    entity_props = defaultdict(set)
    for subj, pred in rows:
        entity_props[subj].add(normalize_schema_type(pred))

    total_entities = con.execute("""
        SELECT COUNT(DISTINCT subject) FROM entity_types
        WHERE type = ? OR type = ? OR type = ? OR type = ?
    """, so_prefix).fetchone()[0]

    set_counter = Counter()
    for subj, props in entity_props.items():
        set_counter[frozenset(props)] += 1
    no_props = total_entities - len(entity_props)
    if no_props > 0:
        set_counter[frozenset()] += no_props

    total = total_entities or 1
    print(f"\n{'─'*60}")
    print(f"  {label} — Characteristic Sets (top {top_n})")
    print(f"{'─'*60}")
    print(f"  Unique property sets : {len(set_counter):,}")
    print()
    for i, (prop_set, count) in enumerate(set_counter.most_common(top_n), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no SO properties)"
        print(f"  #{i:>2}  [{count:>8,}  {pct:>5.1f}%]  {{{props_str}}}")
    return set_counter

for cls, _ in valid_types.most_common(20):
    characteristic_sets_report(con, cls)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  ImageObject — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 73

  # 1  [ 776,975   44.5%]  {contentUrl, height, url, width}
  # 2  [ 521,243   29.8%]  {height, url, width}
  # 3  [ 190,287   10.9%]  {contentUrl, height, representativeOfPage, url, width}
  # 4  [ 109,237    6.3%]  {url}
  # 5  [  31,974    1.8%]  {author, contentUrl, height, name, url, width}
  # 6  [  19,188    1.1%]  {contentUrl, dateCreated, dateModified, height, sourceOrganization, width}
  # 7  [  15,986    0.9%]  {author, contentUrl, datePublished, description}
  # 8  [  10,710    0.6%]  {caption, contentUrl, creditText, dateCreated, dateModified, height, sourceOrganization, width}
  # 9  [  10,410    0.6%]  {creditText, height, url, width}
  #10  [   9,987    0.6%]  {caption, contentUrl, dateCreated, dateModified, height, sourceOrganization, width}
  #11  [   6,655    0.4%]  {conten

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  ListItem — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 18

  # 1  [ 533,614   35.4%]  {item, name, position}
  # 2  [ 465,319   30.8%]  {position, url}
  # 3  [ 288,291   19.1%]  {item, name, position, url}
  # 4  [ 194,354   12.9%]  {item, position}
  # 5  [   9,978    0.7%]  {name, position}
  # 6  [   9,921    0.7%]  {image, name, position, url}
  # 7  [   7,125    0.5%]  {name, position, url}
  # 8  [     314    0.0%]  {additionalType, image, name, position, url}
  # 9  [     145    0.0%]  {name, position, previousItem}
  #10  [      19    0.0%]  {description, name, position, url}
  #11  [       9    0.0%]  {item}
  #12  [       4    0.0%]  {(no SO properties)}
  #13  [       2    0.0%]  {item, name, nextItem, position}
  #14  [       1    0.0%]  {bestRating, item, name, position, ratingCount, ratingValue, worstRating}
  #15  [       1    0.0%]  {ad

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  SportsTeam — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 45

  # 1  [ 919,876   88.0%]  {name}
  # 2  [  70,034    6.7%]  {image, name, url}
  # 3  [  40,922    3.9%]  {image, name}
  # 4  [   4,394    0.4%]  {memberOf, name, sport}
  # 5  [   3,228    0.3%]  {(no SO properties)}
  # 6  [   2,146    0.2%]  {name, url}
  # 7  [   2,093    0.2%]  {athlete, name}
  # 8  [     548    0.1%]  {athlete, memberOf, name, sport, url}
  # 9  [     514    0.0%]  {coach, foundingDate, foundingLocation, name, sport}
  #10  [     325    0.0%]  {athlete}
  #11  [     305    0.0%]  {, name}
  #12  [     216    0.0%]  {memberOf, name, sport, url}
  #13  [     163    0.0%]  {athlete, coach, memberOf, name, sport, url}
  #14  [     144    0.0%]  {image, name, sport, url}
  #15  [     132    0.0%]  {name, sport}
  #16  [     123    0.0%]  {logo, name, sport, url}
  #17  [  

... (14 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  NewsMediaOrganization — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 45

  # 1  [  77,283   41.8%]  {logo, name}
  # 2  [  39,033   21.1%]  {logo, name, url}
  # 3  [  30,599   16.6%]  {actionableFeedbackPolicy, address, contactPoint, correctionsPolicy, diversityPolicy, ethicsPolicy, foundingDate, logo, masthead, missionCoveragePrioritiesPolicy, name, publishingPrinciples, unnamedSourcesPolicy, url, verificationFactCheckingPolicy}
  # 4  [  30,356   16.4%]  {foundingDate, logo, name, sameAs, url}
  # 5  [   4,395    2.4%]  {description, email, location, logo, name, url}
  # 6  [     534    0.3%]  {alternateName, foundingDate, logo, name, parentOrganization, sameAs, url}
  # 7  [     352    0.2%]  {description, email, logo, name, sameAs, url}
  # 8  [     328    0.2%]  {name, url}
  # 9  [     305    0.2%]  {address, contactPoint, founder, foundingDate, l

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


────────────────────────────────────────────────────────────
  Article — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 133

  # 1  [  49,185   28.5%]  {author, dateModified, datePublished, headline, image, mainEntity, name, publisher, sameAs, url}
  # 2  [  17,949   10.4%]  {author, dateModified, datePublished, headline, mainEntity, name, publisher, sameAs, url}
  # 3  [  16,179    9.4%]  {alternativeHeadline, associatedMedia, author, copyrightHolder, copyrightYear, creator, dateCreated, dateModified, datePublished, description, discussionUrl, genre, headline, image, inLanguage, keywords, license, mainEntityOfPage, name, publisher, publishingPrinciples, sourceOrganization, speakable, thumbnailUrl, url}
  # 4  [  14,293    8.3%]  {about, alternativeHeadline, articleBody, articleSection, associatedMedia, author, citation, contentLocation, copyrightHolder, copyrightYear, creator, dateCreated, dateModified, datePublished

## 12. Nesting depth

In [ ]:
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean", "Date", "DateTime", "Time", "URL", "Duration"}

def nesting_depth_report(con, label):
    # Load entity-to-entity reference graph into Python dict (small: ~690k rows)
    print("Loading entity refs …")
    refs = defaultdict(set)
    for subj, linked in con.execute("SELECT subject, linked_subject FROM entity_refs").fetchall():
        refs[subj].add(linked)
    all_subjects = set(con.execute("SELECT DISTINCT subject FROM triples").df()["subject"])

    # Compute max depth with cycle-safe DFS
    memo = {}
    def _max_depth(key, visiting):
        if key in memo: return memo[key]
        if key not in all_subjects or key in visiting: return 0
        visiting.add(key)
        best = max((_max_depth(r, visiting) + 1 for r in refs.get(key, set())), default=0)
        visiting.discard(key)
        memo[key] = best
        return best

    print(f"Computing depths for {len(all_subjects):,} entities …")
    depths_map = {s: _max_depth(s, set()) for s in all_subjects}
    depths = list(depths_map.values())

    n   = len(depths)
    avg = statistics.mean(depths)
    med = statistics.median(depths)
    mx  = max(depths)
    std = statistics.stdev(depths) if n > 1 else 0.0
    srt = sorted(depths)
    p25, p75 = srt[int(n*0.25)], srt[int(n*0.75)]
    p90, p95 = srt[int(n*0.90)], srt[min(int(n*0.95), n-1)]
    depth_counts = Counter(depths)

    print(f"\n{'═'*62}")
    print(f"  {label} — Nesting Depth Analysis")
    print(f"{'═'*62}")
    print(f"  Entities analysed : {n:>10,}")
    print(f"  Mean depth        : {avg:>10.2f}  (σ = {std:.2f})")
    print(f"  Median depth      : {med:>10.1f}")
    print(f"  Max depth         : {mx:>10}")
    print(f"\n  Percentiles:  p25={p25}  p75={p75}  p90={p90}  p95={p95}")

    print(f"\n  Depth distribution:")
    bar_max = max(depth_counts.values(), default=1)
    for d in range(0, mx + 1):
        cnt = depth_counts.get(d, 0)
        bar = '█' * (cnt * 40 // bar_max)
        print(f"    depth {d:>2}: {bar:<40} {cnt:>9,}  ({cnt/n*100:5.1f}%)")

    flat_n   = depth_counts.get(0, 0)
    nested_n = n - flat_n
    deep_n   = sum(v for d, v in depth_counts.items() if d >= 3)
    print(f"\n  Flatness:")
    print(f"    Flat   (depth = 0) : {flat_n:>10,}  ({flat_n/n*100:.1f}%)")
    print(f"    Nested (depth ≥ 1) : {nested_n:>10,}  ({nested_n/n*100:.1f}%)")
    print(f"    Deep   (depth ≥ 3) : {deep_n:>10,}  ({deep_n/n*100:.1f}%)")

    # Cross-reference density
    ref_rows = con.execute("""
        SELECT subject,
               SUM(CASE WHEN linked_subject LIKE '_:%' THEN 1 ELSE 0 END) AS bn,
               SUM(CASE WHEN linked_subject NOT LIKE '_:%' THEN 1 ELSE 0 END) AS uri
        FROM entity_refs GROUP BY subject
    """).fetchall()
    total_bn  = sum(r[1] for r in ref_rows)
    total_uri = sum(r[2] for r in ref_rows)
    total_refs = total_bn + total_uri
    print(f"\n  Cross-reference density:")
    print(f"    Total cross-refs    : {total_refs:>10,}")
    print(f"    Avg refs per entity : {total_refs/n:>10.2f}")
    if total_refs:
        print(f"    Blank-node refs     : {total_bn:>10,}  ({total_bn/total_refs*100:.1f}%)")
        print(f"    URI refs            : {total_uri:>10,}  ({total_uri/total_refs*100:.1f}%)")

    # Types of directly referenced entities
    ref_type_counts = Counter(dict(con.execute("""
        SELECT normalize_type, COUNT(*) AS cnt
        FROM (
            SELECT regexp_replace(et.type,
                '^https?://(www[.])?schema[.]org/', '') AS normalize_type
            FROM entity_refs er
            JOIN entity_types et ON er.linked_subject = et.subject
            WHERE et.type LIKE 'http://schema.org/%' OR et.type LIKE 'https://schema.org/%'
        ) GROUP BY normalize_type ORDER BY cnt DESC LIMIT 15
    """).fetchall()))
    if ref_type_counts:
        print(f"\n  Types of directly referenced entities (top 15):")
        for t, c in ref_type_counts.most_common(15):
            print(f"    {t:<35} {c:>9,}")

    # Properties linking to structured vs literal values
    prop_struct = con.execute("""
        SELECT regexp_replace(ep.predicate, '^https?://(www[.])?schema[.]org/', '') AS pname,
               SUM(CASE WHEN ep.object LIKE '<%>' OR ep.object LIKE '_:%' THEN 1 ELSE 0 END) AS struct_cnt,
               SUM(CASE WHEN ep.object NOT LIKE '<%>' AND ep.object NOT LIKE '_:%' THEN 1 ELSE 0 END) AS lit_cnt
        FROM entity_pred ep
        WHERE ep.predicate LIKE 'http://schema.org/%' OR ep.predicate LIKE 'https://schema.org/%'
        GROUP BY pname HAVING struct_cnt + lit_cnt >= 50
        ORDER BY struct_cnt::FLOAT / (struct_cnt + lit_cnt) DESC LIMIT 15
    """).fetchall()
    print(f"\n  Properties most often linking to structured entities:")
    print(f"  {'property':<25} {'literal':>8} {'struct':>8} {'%struct':>8}")
    for pname, struct, lit in prop_struct:
        total = lit + struct
        print(f"  {pname:<25} {lit:>8,} {struct:>8,} {struct/total*100:>7.1f}%")

    # Literal-only enrichment candidates
    lit_only = con.execute("""
        SELECT regexp_replace(ep.predicate, '^https?://(www[.])?schema[.]org/', '') AS pname,
               COUNT(*) AS uses
        FROM entity_pred ep
        WHERE (ep.predicate LIKE 'http://schema.org/%' OR ep.predicate LIKE 'https://schema.org/%')
          AND ep.object NOT LIKE '<%>' AND ep.object NOT LIKE '_:%'
          AND NOT EXISTS (
              SELECT 1 FROM entity_pred ep2
              WHERE ep2.predicate = ep.predicate
                AND (ep2.object LIKE '<%>' OR ep2.object LIKE '_:%')
          )
        GROUP BY pname ORDER BY uses DESC LIMIT 15
    """).fetchall()
    print(f"\n  Literal-only enrichment candidates")
    print(f"  {'property':<25} {'uses':>8}  entity range types")
    for pname, uses in lit_only:
        er = ", ".join(sorted((so_prop_range.get(pname, set()) - LITERAL_DATATYPES))[:4])
        if er:
            print(f"  {pname:<25} {uses:>8,}  {er}")

    return depths_map

depths_map = nesting_depth_report(con, "All entities (merged.nq)")


Loading entity refs …


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Computing depths for 12,211,353 entities …


## 13. Deeply nested entities — examples

In [ ]:
# Top 5 deepest entities
top5 = sorted(depths_map.items(), key=lambda x: x[1], reverse=True)[:5]

for subj, depth in top5:
    types_row = con.execute("""
        SELECT type FROM entity_types WHERE subject = ?
    """, [subj]).fetchall()
    types_str = ", ".join(normalize_schema_type(r[0]) for r in types_row) or "untyped"
    graphs_row = con.execute("SELECT graph FROM entity_graphs WHERE subject = ?", [subj]).fetchall()

    print(f"\n{'═'*70}")
    print(f"  subject : {subj}")
    print(f"  types   : {types_str}")
    print(f"  depth   : {depth}")
    print(f"  graphs  : {set(r[0] for r in graphs_row)}")
    print(f"  props:")

    props_row = con.execute("""
        SELECT predicate, object FROM entity_pred WHERE subject = ?
    """, [subj]).fetchall()
    for pred, obj in props_row:
        pname = normalize_schema_type(pred)
        # Check if obj is a linked entity
        linked_subj = obj[1:-1] if (obj.startswith('<') and obj.endswith('>')) else (obj if obj.startswith('_:') else None)
        if linked_subj:
            child_types = con.execute("SELECT type FROM entity_types WHERE subject = ?", [linked_subj]).fetchall()
            child_type_str = ", ".join(normalize_schema_type(r[0]) for r in child_types) or "untyped"
            print(f"    {pname:<25} → [{child_type_str}]  {linked_subj[:60]}")
            # One level deeper
            child_props = con.execute("""
                SELECT predicate, object FROM entity_pred WHERE subject = ?
            """, [linked_subj]).fetchall()
            for pred2, obj2 in child_props:
                pname2 = normalize_schema_type(pred2)
                linked2 = obj2[1:-1] if (obj2.startswith('<') and obj2.endswith('>')) else (obj2 if obj2.startswith('_:') else None)
                if linked2:
                    gc_types = con.execute("SELECT type FROM entity_types WHERE subject = ?", [linked2]).fetchall()
                    gc_type_str = ", ".join(normalize_schema_type(r[0]) for r in gc_types) or "untyped"
                    print(f"      {pname2:<23} → [{gc_type_str}]  {linked2[:58]}")
                else:
                    print(f"      {pname2:<23} = {obj2[:78]}")
        else:
            print(f"    {pname:<25} = {obj[:80]}")



══════════════════════════════════════════════════════════════════════
  subject : _:node1ife5c4iox46188140
  types   : ListItem
  depth   : 25
  graphs  : {'https://resultados.as.com/resultados/futbol/peru_apertura/2023/jornada/regular_a_18/'}
  props:
    position                  = "0"@es
    name                      = "Resultados"@es
    item                      → [untyped]  https://resultados.as.com/resultados/futbol/peru_apertura/20
      http://www.w3.org/1999/xhtml/microdata#item → [SportsTeam]  _:node1ife5c4iox46188161
      http://www.w3.org/1999/xhtml/microdata#item → [SportsTeam]  _:node1ife5c4iox46188177
      http://www.w3.org/1999/xhtml/microdata#item → [SportsTeam]  _:node1ife5c4iox46188153
      http://www.w3.org/1999/xhtml/microdata#item → [SportsTeam]  _:node1ife5c4iox46188146
      http://www.w3.org/1999/xhtml/microdata#item → [StadiumOrArena]  _:node1ife5c4iox46188171
      http://www.w3.org/1999/xhtml/microdata#item → [StadiumOrArena]  _:node1ife5c4iox46188159


In [ ]:
# _parse_host is defined in the avg-types-per-page cell above


In [ ]:
# # ── Schema.org property fill rate per entity, page, and domain ───

# # Build entity → (available_props, used_props) using vocab and DuckDB
# @lru_cache(maxsize=None)
# def _allowed(types_frozen):
#     allowed = set()
#     for t in types_frozen:
#         allowed |= so_type_props.get(t, set())
#     return allowed

# # Fetch entity types
# subj_types = defaultdict(set)
# for subj, t in con.execute("""
#     SELECT subject, type FROM entity_types
#     WHERE type LIKE 'http://schema.org/%' OR type LIKE 'https://schema.org/%'
# """).fetchall():
#     subj_types[subj].add(normalize_schema_type(t))

# # Fetch entity used SO props (distinct per entity)
# subj_used = defaultdict(set)
# for subj, pred in con.execute("""
#     SELECT DISTINCT subject, predicate FROM entity_pred
#     WHERE predicate LIKE 'http://schema.org/%' OR predicate LIKE 'https://schema.org/%'
# """).fetchall():
#     subj_used[subj].add(normalize_schema_type(pred))

# # Fetch entity graphs
# subj_graphs = defaultdict(set)
# for subj, graph in con.execute("SELECT subject, graph FROM entity_graphs").fetchall():
#     subj_graphs[subj].add(graph)

# # Build fill records
# fill_records = []
# for subj, types in subj_types.items():
#     available = _allowed(frozenset(types))
#     if not available:
#         continue
#     used = subj_used.get(subj, set()) & available
#     fill_rate = len(used) / len(available)
#     for graph in subj_graphs.get(subj, set()):
#         _, reg = _parse_host(graph)
#         fill_records.append({
#             "subj":              subj,
#             "uri":               graph,
#             "registered_domain": reg,
#             "primary_type":      next(iter(sorted(types))),
#             "available":         len(available),
#             "used":              len(used),
#             "fill_rate":         fill_rate,
#         })

# df_fill = pd.DataFrame(fill_records)
# print(f"Entity-page pairs: {len(df_fill):,}")
# print(f"Mean fill rate overall: {df_fill['fill_rate'].mean()*100:.1f}%\n")

# print(f"\n{'='*65}")
# print("Fill rate per registered domain (avg across all entities)")
# print(f"{'='*65}")
# print(f"  {'domain':<35} {'entities':>9}  {'avail':>6}  {'mean_used':>10}  {'fill%':>6}")
# print(f"  {'─'*70}")

# domain_fill = (
#     df_fill.groupby("registered_domain")
#     .agg(
#         entities  = ("subj",      "nunique"),
#         avg_avail = ("available", "mean"),
#         avg_used  = ("used",      "mean"),
#         fill_rate = ("fill_rate", "mean"),
#     )
#     .sort_values("entities", ascending=False)
# )
# for domain, row in domain_fill.iterrows():
#     print(f"  {domain:<35} {row['entities']:>9,}  "
#           f"{row['avg_avail']:>6.0f}  {row['avg_used']:>10.2f}  "
#           f"{row['fill_rate']*100:>5.1f}%")

# print(f"\n{'='*60}")
# print("Fill rate distribution (all entity-page pairs)")
# print(f"{'='*60}")
# buckets = pd.cut(
#     df_fill["fill_rate"],
#     bins=[0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.001],
#     include_lowest=True,
#     labels=["0–5%", "5–10%", "10–20%", "20–30%", "30–50%", "50–100%"],
# )
# dist = buckets.value_counts().sort_index()
# n = dist.sum()
# for lbl, count in dist.items():
#     bar = "█" * (count * 40 // dist.max())
#     print(f"  {lbl:<10} {bar:<40} {count:>9,}  ({count/n*100:>5.1f}%)")

# print(f"\n{'='*60}")
# print("Most completely annotated entities (top 10 by fill rate, min 5 used props)")
# print(f"{'='*60}")
# top = df_fill[df_fill["used"] >= 5].nlargest(10, "fill_rate")
# for _, row in top.iterrows():
#     uri = row["uri"][:70] + "…" if len(row["uri"]) > 70 else row["uri"]
#     print(f"  {row['fill_rate']*100:>5.1f}%  {row['used']:>3}/{row['available']:<3}  [{row['primary_type']}]  {uri}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# ── Schema.org property fill rate (memory-safe) ────────────────
# Spill-to-disk lets DuckDB handle datasets that exceed available RAM.
import os; os.makedirs('/tmp/duckdb_spill', exist_ok=True)
con.execute("SET threads = 2")
con.execute("SET memory_limit = '3GB'")
con.execute("SET temp_directory = '/tmp/duckdb_spill'")
con.execute("SET preserve_insertion_order = false")

# [1/4] type→prop mapping (small table)
_tp_df = pd.DataFrame(
    [(t, p) for t, props in so_type_props.items() for p in props],
    columns=["type_local", "prop_local"]
)
con.execute("CREATE OR REPLACE TABLE _so_tp AS SELECT * FROM _tp_df")
print(f"[1/4] type→prop pairs: {len(_tp_df):,}")

# [2/4] entity types — canonical (schema.org) and non-canonical (www.schema.org)
con.execute("""
CREATE OR REPLACE TABLE _etype AS
SELECT subject, regexp_extract(type, '[^/]+$') AS type_local
FROM entity_types
WHERE type LIKE 'http://schema.org/%'     OR type LIKE 'https://schema.org/%'
   OR type LIKE 'http://www.schema.org/%' OR type LIKE 'https://www.schema.org/%'
""")
print(f"[2/4] entity-type rows: {con.execute('SELECT COUNT(*) FROM _etype').fetchone()[0]:,}")

# [3/4] SO predicates per entity — canonical and non-canonical
con.execute("""
CREATE OR REPLACE TABLE _epred AS
SELECT DISTINCT subject, regexp_extract(predicate, '[^/]+$') AS pred_local
FROM entity_pred
WHERE predicate LIKE 'http://schema.org/%'     OR predicate LIKE 'https://schema.org/%'
   OR predicate LIKE 'http://www.schema.org/%' OR predicate LIKE 'https://www.schema.org/%'
""")
print(f"[3/4] distinct entity-pred pairs: {con.execute('SELECT COUNT(*) FROM _epred').fetchone()[0]:,}")

# [4/4] per-entity available/used stats
# Inline 3-way join avoids materialising the full subjects×props cross-product
con.execute("""
CREATE OR REPLACE TABLE _entity_stats AS
SELECT
    e.subject,
    MIN(e.type_local)             AS primary_type,
    COUNT(DISTINCT s.prop_local)  AS available,
    COUNT(DISTINCT ep.pred_local) AS used
FROM      _etype  e
JOIN      _so_tp  s  ON e.type_local  = s.type_local
LEFT JOIN _epred  ep ON e.subject     = ep.subject AND ep.pred_local = s.prop_local
GROUP BY e.subject
""")
for t in ["_so_tp", "_etype", "_epred"]:
    con.execute(f"DROP TABLE IF EXISTS {t}")

n_entities = con.execute("SELECT COUNT(*) FROM _entity_stats").fetchone()[0]
mean_fill  = con.execute("SELECT AVG(used::DOUBLE / available) FROM _entity_stats").fetchone()[0]
print(f"[4/4] entities with types: {n_entities:,}")
print(f"\nTotal entities : {n_entities:,}")
print(f"Mean fill rate : {mean_fill*100:.1f}%\n")

# ── Per-domain fill rate (aggregated in SQL) ─────────────────────
df_domain = con.execute("""
SELECT
    regexp_replace(
        regexp_extract(eg.graph, '^https?://([^/]+)', 1),
        '^www\.', ''
    )                                        AS domain,
    COUNT(DISTINCT es.subject)               AS entities,
    AVG(es.available)                        AS avg_avail,
    AVG(es.used)                             AS avg_used,
    AVG(es.used::DOUBLE / es.available)      AS fill_rate
FROM _entity_stats es
JOIN entity_graphs eg ON es.subject = eg.subject
GROUP BY domain
ORDER BY entities DESC
""").df()

print(f"\n{'='*65}")
print("Fill rate per registered domain (avg across all entities)")
print(f"{'='*65}")
print(f"  {'domain':<35} {'entities':>9}  {'avail':>6}  {'mean_used':>10}  {'fill%':>6}")
print(f"  {'─'*70}")
for _, row in df_domain.iterrows():
    print(f"  {str(row['domain']):<35} {int(row['entities']):>9,}  "
          f"{row['avg_avail']:>6.0f}  {row['avg_used']:>10.2f}  "
          f"{row['fill_rate']*100:>5.1f}%")

# ── Fill rate distribution ────────────────────────────────────────
dist_rows = con.execute("""
SELECT bucket, COUNT(*) AS cnt FROM (
    SELECT CASE
        WHEN used::DOUBLE / available <= 0.05  THEN '0\u20135%'
        WHEN used::DOUBLE / available <= 0.10  THEN '5\u201310%'
        WHEN used::DOUBLE / available <= 0.20  THEN '10\u201320%'
        WHEN used::DOUBLE / available <= 0.30  THEN '20\u201330%'
        WHEN used::DOUBLE / available <= 0.50  THEN '30\u201350%'
        ELSE '50\u2013100%'
    END AS bucket
    FROM _entity_stats
) GROUP BY bucket ORDER BY bucket
""").fetchall()

dist  = {lbl: cnt for lbl, cnt in dist_rows}
n_tot = sum(dist.values())
bmax  = max(dist.values()) if dist else 1
print(f"\n{'='*60}")
print("Fill rate distribution (all entities)")
print(f"{'='*60}")
for lbl in ["0\u20135%", "5\u201310%", "10\u201320%", "20\u201330%", "30\u201350%", "50\u2013100%"]:
    cnt = dist.get(lbl, 0)
    bar = "\u2588" * (cnt * 40 // bmax)
    print(f"  {lbl:<10} {bar:<40} {cnt:>9,}  ({cnt/n_tot*100:>5.1f}%)")

# ── Top annotated entities ────────────────────────────────────────
top_rows = con.execute("""
SELECT es.primary_type, es.available, es.used,
       es.used::DOUBLE / es.available AS fill_rate,
       eg.graph AS uri
FROM _entity_stats es
JOIN entity_graphs eg ON es.subject = eg.subject
WHERE es.used >= 5
ORDER BY fill_rate DESC
LIMIT 10
""").fetchall()

print(f"\n{'='*60}")
print("Most completely annotated entities (top 10, min 5 used props)")
print(f"{'='*60}")
for ptype, avail, used, fr, uri in top_rows:
    uri_disp = uri[:70] + "\u2026" if len(uri) > 70 else uri
    print(f"  {fr*100:>5.1f}%  {used:>3}/{avail:<3}  [{ptype}]  {uri_disp}")

con.execute("DROP TABLE IF EXISTS _entity_stats")


[1/6] type→prop pairs: 63,436


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2/6] entity-type rows: 11,864,977


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 128.0 KiB (15.3 GiB/15.3 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [ ]:
# ── Co-occurring entity types on pages with Article or NewsArticle ──

ARTICLE_TYPES = {"Article", "NewsArticle"}

# Materialize article pages ONCE — replaces all the repeated correlated subqueries
con.execute("""
CREATE OR REPLACE TEMP TABLE _article_pages AS
SELECT DISTINCT eg.graph
FROM entity_graphs eg
JOIN entity_types et ON eg.subject = et.subject
WHERE et.type IN (
    'http://schema.org/Article',    'https://schema.org/Article',
    'http://schema.org/NewsArticle','https://schema.org/NewsArticle'
)
""")

target_pages = set(con.execute("SELECT graph FROM _article_pages").df()["graph"])

article_only = set(con.execute("""
    SELECT DISTINCT eg.graph FROM entity_graphs eg
    JOIN entity_types et ON eg.subject = et.subject
    WHERE et.type IN ('http://schema.org/Article','https://schema.org/Article')
""").df()["graph"])

news_only = set(con.execute("""
    SELECT DISTINCT eg.graph FROM entity_graphs eg
    JOIN entity_types et ON eg.subject = et.subject
    WHERE et.type IN ('http://schema.org/NewsArticle','https://schema.org/NewsArticle')
""").df()["graph"])

print(f"Pages with Article OR NewsArticle: {len(target_pages):,}")
print(f"  Article pages     : {len(article_only):,}")
print(f"  NewsArticle pages : {len(news_only):,}")
print(f"  Overlap (both)    : {len(article_only & news_only):,}")

# JOIN against temp table instead of correlated subquery
co_type_rows = con.execute("""
    SELECT
        regexp_replace(et.type, '^https?://(www[.])?schema[.]org/', '') AS local_type,
        COUNT(DISTINCT eg.graph) AS page_cnt
    FROM entity_graphs  eg
    JOIN entity_types   et ON eg.subject = et.subject
    JOIN _article_pages ap ON eg.graph   = ap.graph
    WHERE et.type LIKE 'http://schema.org/%' OR et.type LIKE 'https://schema.org/%'
    GROUP BY local_type ORDER BY page_cnt DESC LIMIT 25
""").fetchall()

n_target = len(target_pages)
print(f"\n{'='*65}")
print("Co-occurring entity types on Article+NewsArticle pages")
print(f"{'='*65}")
print(f"  {'type':<35} {'pages':>8}  {'% of target pages':>18}")
print(f"  {'─'*63}")
co_type_counter = Counter()
for local_type, page_cnt in co_type_rows:
    co_type_counter[local_type] = page_cnt
    marker = " ◄ article type" if local_type in ARTICLE_TYPES else ""
    print(f"  {local_type:<35} {page_cnt:>8,}  {page_cnt/n_target*100:>17.1f}%{marker}")

TOP_CO_TYPES = [t for t, _ in co_type_counter.most_common(20) if t not in ARTICLE_TYPES]

print(f"\n\n{'='*65}")
print("Characteristic property sets — Article, NewsArticle & top co-occurring types")
print("(showing how richly each type is annotated)")
print(f"{'='*65}")

for cls in ["Article", "NewsArticle"] + TOP_CO_TYPES:
    so_http  = f"http://schema.org/{cls}"
    so_https = f"https://schema.org/{cls}"

    # string_agg returns ONE row per entity instead of one row per (entity, predicate)
    # e.g. 100k entities × 10 props = 1M rows → now 100k rows
    entity_rows = con.execute("""
        SELECT ep.subject,
               string_agg(DISTINCT regexp_extract(ep.predicate, '[^/]+$'), ',') AS props
        FROM entity_types   et
        JOIN entity_pred     ep ON et.subject = ep.subject
        JOIN entity_graphs   eg ON et.subject = eg.subject
        JOIN _article_pages  ap ON eg.graph   = ap.graph
        WHERE (et.type = ? OR et.type = ?)
          AND (ep.predicate LIKE 'http://schema.org/%' OR ep.predicate LIKE 'https://schema.org/%')
        GROUP BY ep.subject
    """, (so_http, so_https)).fetchall()

    total_cls = con.execute("""
        SELECT COUNT(DISTINCT et.subject)
        FROM entity_types et
        JOIN entity_graphs  eg ON et.subject = eg.subject
        JOIN _article_pages ap ON eg.graph   = ap.graph
        WHERE et.type = ? OR et.type = ?
    """, (so_http, so_https)).fetchone()[0]

    set_counter = Counter()
    for _, props_str in entity_rows:
        prop_set = frozenset(props_str.split(',')) if props_str else frozenset()
        set_counter[prop_set] += 1

    no_props = total_cls - sum(set_counter.values())
    if no_props > 0:
        set_counter[frozenset()] += no_props

    avp       = sum(len(ps) * cnt for ps, cnt in set_counter.items()) / total_cls if total_cls else 0
    empty_pct = set_counter.get(frozenset(), 0) / total_cls * 100
    n_avail   = len(so_type_props.get(cls, set()))

    print(f"\n{'─'*65}")
    print(f"  {cls}  ({total_cls:,} entities | avail={n_avail} props | AvP={avp:.1f} | no-prop={empty_pct:.1f}%)")
    print(f"{'─'*65}")
    for i, (prop_set, count) in enumerate(set_counter.most_common(10), 1):
        pct = count / total_cls * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no schema.org properties)"
        print(f"  #{i:>2}  [{count:>7,}  {pct:>5.1f}%]  {{{props_str}}}")

# page_to_subjects: restrict to article pages only — ALL pages would be tens of millions of rows
print("\nBuilding page_to_subjects index (article pages only) …")
page_to_subjects = defaultdict(set)
for subj, graph in con.execute("""
    SELECT eg.subject, eg.graph
    FROM entity_graphs eg
    JOIN _article_pages ap ON eg.graph = ap.graph
""").fetchall():
    page_to_subjects[graph].add(subj)
print(f"  {len(page_to_subjects):,} article pages indexed")

con.execute("DROP TABLE IF EXISTS _article_pages")


Pages with Article OR NewsArticle: 356,132
  Article pages     : 169,598
  NewsArticle pages : 193,341
  Overlap (both)    : 6,807


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Co-occurring entity types on Article+NewsArticle pages
  type                                   pages   % of target pages
  ───────────────────────────────────────────────────────────────
  ImageObject                          355,867               99.9%
  Organization                         311,631               87.5%
  NewsArticle                          193,341               54.3% ◄ article type
  ListItem                             176,595               49.6%
  BreadcrumbList                       176,317               49.5%
  WebPage                              172,518               48.4%
  Article                              169,598               47.6% ◄ article type
  PostalAddress                        156,715               44.0%
  Person                               147,760               41.5%
  WebSite                              120,760               33.9%
  NewsMediaOrganization                107,621               30.2%
  AggregateRating                       96,4

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


─────────────────────────────────────────────────────────────────
  Article  (172,671 entities | avail=136 props | AvP=19.6 | no-prop=0.0%)
─────────────────────────────────────────────────────────────────
  # 1  [ 49,185   28.5%]  {author, dateModified, datePublished, headline, image, mainEntity, name, publisher, sameAs, url}
  # 2  [ 17,949   10.4%]  {author, dateModified, datePublished, headline, mainEntity, name, publisher, sameAs, url}
  # 3  [ 16,179    9.4%]  {alternativeHeadline, associatedMedia, author, copyrightHolder, copyrightYear, creator, dateCreated, dateModified, datePublished, description, discussionUrl, genre, headline, image, inLanguage, keywords, license, mainEntityOfPage, name, publisher, publishingPrinciples, sourceOrganization, speakable, thumbnailUrl, url}
  # 4  [ 14,293    8.3%]  {about, alternativeHeadline, articleBody, articleSection, associatedMedia, author, citation, contentLocation, copyrightHolder, copyrightYear, creator, dateCreated, dateModified, date

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


─────────────────────────────────────────────────────────────────
  NewsMediaOrganization  (179,024 entities | avail=93 props | AvP=5.0 | no-prop=0.0%)
─────────────────────────────────────────────────────────────────
  # 1  [ 74,937   41.9%]  {logo, name}
  # 2  [ 37,722   21.1%]  {logo, name, url}
  # 3  [ 30,170   16.9%]  {foundingDate, logo, name, sameAs, url}
  # 4  [ 29,686   16.6%]  {actionableFeedbackPolicy, address, contactPoint, correctionsPolicy, diversityPolicy, ethicsPolicy, foundingDate, logo, masthead, missionCoveragePrioritiesPolicy, name, publishingPrinciples, unnamedSourcesPolicy, url, verificationFactCheckingPolicy}
  # 5  [  4,230    2.4%]  {description, email, location, logo, name, url}
  # 6  [    534    0.3%]  {alternateName, foundingDate, logo, name, parentOrganization, sameAs, url}
  # 7  [    352    0.2%]  {description, email, logo, name, sameAs, url}
  # 8  [    328    0.2%]  {name, url}
  # 9  [    305    0.2%]  {address, contactPoint, founder, foundingDate

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  356,132 article pages indexed


In [ ]:
# make sure the pages are actually articles/news articles

absence of event indicates that articles are not describing the actual event happening 